# Code for Dependence

In [89]:
#@title Setup

import sympy as sp
import numpy as np
import itertools as it
import pickle
from scipy.stats import qmc, uniform
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

plt.rcParams['svg.fonttype'] = 'none'

# format = 'presentation'
format = 'paper'

if format == "paper":
    plt.rcParams.update({
        "font.size": 10,
        "axes.labelsize": 11,
        "axes.titlesize": 13,
        "figure.labelsize": 14,
        "figure.titlesize": 14,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 10,
        "lines.linewidth": 1.8,
        "figure.dpi": 150,
        "savefig.dpi": 300,
    })

elif format == "presentation":
    plt.rcParams.update({
        "font.size": 12,
        "axes.labelsize": 13,
        "axes.titlesize": 15,
        "figure.labelsize": 16,
        "figure.titlesize": 15,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 12,
        "lines.linewidth": 2.5,
        "axes.linewidth": 1.2,
        "xtick.major.width": 1.2,
        "ytick.major.width": 1.2,
        "xtick.major.size": 5,
        "ytick.major.size": 5,
        "figure.dpi": 150,
        "savefig.dpi": 300,
    })


# Traits

x, X = sp.symbols('x X')  # Focal's trait values
y, Y = sp.symbols('y Y')  # Other individuals' trait values
z, Z = sp.symbols('z Z')  # Global population mean trait values

# Parameters

Sq = sp.Symbol('S_q')  # Queen survival
Sqh = sp.Symbol('S_qh')  # Queen-helper joint survival
Sh = sp.Symbol('S_h')  # Helper reproductive opportunity
Shh = sp.Symbol('S_hh')  # Helper-helper joint survival
n = sp.Symbol('n', integer=True, positive=True)  # Number of helpers
r = sp.Symbol('r')  # Relatedness
c = sp.Symbol('c')  # Cost of helping
P0 = sp.Symbol('P_0')  # Baseline productivity
PΔ = sp.Symbol('P_Δ')  # Productivity benefit of specialisation

symbols = [Sh, Shh, Sq, Sqh, c, n, PΔ, P0, r]  # Parameter symbols

resolution = 200

param_ranges = {
    's_season': np.linspace(0.01, 0.99, resolution),
    'c': np.linspace(0.01, 0.99, resolution),
    'r': np.linspace(0.01, 0.99, resolution),
    'n': np.arange(2, 51),
    'T': np.arange(2, 51),
    'Pd': np.linspace(0.01, 10, resolution),
    'P0': np.linspace(0.01, 10, resolution),
}

labels = {
    'n': 'Helper number ($n$)',
    'r': 'Relatedness ($r$)',
    'c': 'Cost of reproductive\npotential to helping ($c$)',
    's_season': 'Seasonal survival ($s^T$)',
    'T': 'Season length ($T$)',
    'P0': 'Baseline productivity ($P_0$)',
    'Pd': r'Productivity benefit ($P_\Delta$)'
    }

endpoint_labels = [
    "unresolved",
    "unspecialised",
    "partial breeder, no helper",
    "no breeder, partial helper",
    "complete breeder, no helper",
    "partial breeder, partial helper",
    "no breeder, complete helper",
    "complete breeder, partial helper",
    "partial breeder, complete helper",
    "complete breeder, complete helper",
]

# Fitness

Wq = P0*(1-X)*Sq + (1 - c*y)*(1 + PΔ*X)*n*Sqh  # Queen fitness
Wh = x*P0*(1-X)*Sh + x*(1 - c*y)*(1 + PΔ*X)*(n - 1)*Shh  # Helper fitness
W = Wq + n*Wh  # Total fitness

# Selection Gradients

WX = (W.diff(X) + r*W.diff(Y)).subs({x: z, y: z, X: Z, Y: Z})  # Queen selection gradient
Wx = (W.diff(x) + r*W.diff(y)).subs({x: z, y: z, X: Z, Y: Z})  # Helper selection gradient

WX_fun = sp.lambdify([z, *symbols], WX, 'numpy')  # Numerical queen selection gradient
Wx_fun = sp.lambdify([z, Z, *symbols], Wx, 'numpy')  # Numerical helper selection gradient

In [90]:

#@title prepare_parameters

def prepare_parameters(samples, grid=False):
    """
    Calculate composite parameters and arrange arrays for vectorised evaluation.

    samples: Dictionary of model parameters, each scalar or array.

    grid=False: Pair array values by index. Each index represents one
                parameter combination. Used for sensitivity analysis.

    grid=True:  Construct the Cartesian product of all array-valued
                parameters, evaluating every possible combination.
                Used for parameter-space plots.

    Returns broadcast arrays of composite and original parameters.
    """

    samples = samples.copy()

    # Construct grid over varying parameters
    varying = [k for k, v in samples.items() if np.ndim(v) > 0]

    if grid and len(varying) > 1:
        arrays = np.meshgrid(*(samples[k] for k in varying), indexing='ij')
        samples.update(zip(varying, arrays))

    n = samples['n']
    T = samples['T']
    c = samples['c']
    r = samples['r']
    s_season = samples['s_season']
    P0 = samples['P0']
    PΔ = samples['Pd']

    # Convert seasonal to per-step survival
    s = s_season**(1 / T)

    # Cumulative survivals
    Sq = s * (1 - s**T) / (1 - s)
    Sqh = s**2 * (1 - s**(2*T)) / (1 - s**2)

    shape = np.broadcast_shapes(*(np.shape(v) for v in samples.values()))

    t = np.arange(1, int(np.max(T)) + 1).reshape(
        (-1,) + (1,) * len(shape)
    )

    inheritance = (
        s**(t - 1) * (1 - s)
        * (1 - (1 - s**t)**n) / n
    )

    remaining = np.maximum(T - t + 1, 0)

    Sh = np.sum(
        np.where(
            t <= T,
            inheritance * (1 - s**remaining) / (1 - s),
            0
        ),
        axis=0
    )

    Shh = np.sum(
        np.where(
            t <= T,
            inheritance * (1 - s**(2*remaining)) / (1 - s**2),
            0
        ),
        axis=0
    )

    return np.broadcast_arrays(Sh, Shh, Sq, Sqh,c, n, PΔ, P0, r)


In [91]:
#@title plot_local_stability

def plot_local_stability(background, axes, z=1, Z=0):

    combinations = list(it.combinations(axes, 2))
    nrows = (len(combinations) + 2) // 3
    ncols = min(3, len(combinations))

    fig, axs = plt.subplots(
        nrows, ncols,
        figsize=(2.7 * ncols, 2.5 * nrows),
        constrained_layout=True,
        squeeze=False
    )

    cmap = ListedColormap([
        "white",
        "#3DA5A8",  # Breeder
        "#E8D15B",  # Helper
        "#D58DB1",  # Both
    ])

    for ax, pair in zip(axs.flat, combinations):

        samples = background.copy()

        for param in pair:
            samples[param] = param_ranges[param]

        params = prepare_parameters(samples, grid=True)

        Wx = Wx_fun(z, Z, *params)
        WX = WX_fun(z, *params)

        grid = np.meshgrid(
            samples[pair[0]], samples[pair[1]], indexing='ij'
        )

        regions = np.zeros(Wx.shape, dtype=int)

        regions[(WX > 0) & (Wx < 0)] = 3
        regions[(WX > 0) & (Wx >= 0)] = 1
        regions[(WX <= 0) & (Wx < 0)] = 2

        ax.pcolormesh(
            *grid, regions,
            cmap=cmap,
            vmin=-0.5,
            vmax=3.5,
            shading="nearest"
        )

        for gradient in [Wx, WX]:
            ax.contour(
                *grid, gradient,
                levels=[0],
                colors="black"
            )

        ax.set_xlabel(labels[pair[0]])
        ax.set_ylabel(labels[pair[1]])

    for ax in axs.flat[len(combinations):]:
        ax.set_visible(False)

    legend_elements = [
        Patch(facecolor="white", edgecolor="black", label="Neither"),
        Patch(facecolor="#3DA5A8", label="Breeder"),
        Patch(facecolor="#E8D15B", label="Helper"),
        Patch(facecolor="#D58DB1", label="Both"),
    ]

    fig.legend(
        handles=legend_elements,
        title="Specialisation favoured",
        loc="center left",
        bbox_to_anchor=(1.01, 0.85)
    )

    return fig, axs

In [92]:
#@title Example local stability

# Baseline parameters
# background = {
#     's_season': 0.94,
#     'c': 0.5,
#     'r': 0.6,
#     'n': 20,
#     'T': 4,
#     'P0': 10,
#     'Pd': 1.4,
# }

# # Single plot, complete joint specialisation
# fig, axs = plot_local_stability_colour(background, ('s_season', 'r'), z=0, Z=1)

# # Multiple plots, unspecialised ancestor
# fig, axs = plot_local_stability_colour(
#     background, ('s_season', 'c', 'r', 'n'), z=1, Z=0
# )

# # Multiple plots, complete joint specialisation
# fig, axs = plot_local_stability_colour(
#     background, ('s_season', 'c', 'r', 'n'), z=0, Z=1
# )

In [93]:
#@title calc_numerical_endpoints

def calc_numerical_endpoints(axes, background, dt=1e-5, grad_tol=1e-5, max_iter=int(1e6)):
    """
    Calculate evolutionary endpoints and realised facilitation
    across two parameter ranges.
    """

    samples = {parameter: param_ranges[parameter] for parameter in axes}
    samples.update({k: v for k, v in background.items() if k not in axes})

    params = prepare_parameters(samples, grid=True)

    z = np.ones(params[0].shape)
    Z = np.zeros(params[0].shape)

    # Initial selection gradients
    Wx_initial = Wx_fun(z, Z, *params)
    WX_initial = WX_fun(z, *params)

    # Track stop condition
    settled = np.zeros(params[0].shape, dtype=bool)

    for _ in range(max_iter):

        if np.all(settled):
            break

        # Active parameter combinations
        p = [value[~settled] for value in params]

        Wx = Wx_fun(z[~settled], Z[~settled], *p)
        WX = WX_fun(z[~settled], *p)

        # Prevent selection from pushing traits beyond boundaries
        Wx[(z[~settled] <= 0) & (Wx < 0)] = 0
        Wx[(z[~settled] >= 1) & (Wx > 0)] = 0

        WX[(Z[~settled] <= 0) & (WX < 0)] = 0
        WX[(Z[~settled] >= 1) & (WX > 0)] = 0

        # Mark equilibria as settled
        equilibrium = np.maximum(np.abs(Wx), np.abs(WX)) < grad_tol

        settled[~settled] = equilibrium

        # Evolve traits
        z[~settled] += dt * Wx[~equilibrium]
        Z[~settled] += dt * WX[~equilibrium]

        np.clip(z, 0, 1, out=z)
        np.clip(Z, 0, 1, out=Z)

    # Final selection gradients at initial trait boundaries
    Wx_final = Wx_fun(np.ones_like(z), Z, *params)
    WX_final = WX_fun(z, *params)

    # Breeder evolution facilitates helper evolution
    breeder_facilitates = (
        (Wx_initial > 0) &
        (Wx_final < 0) &
        settled
    )

    # Helper evolution facilitates breeder evolution
    helper_facilitates = (
        (WX_initial < 0) &
        (WX_final > 0) &
        settled
    )

    return z, Z, settled, breeder_facilitates, helper_facilitates

In [94]:
#@title plot_numerical_endpoint_classification

def plot_numerical_endpoint_classification(
    z, Z, settled, breeder_facilitates, helper_facilitates,
    axes, trait_tol=1e-5, ax=None
):

    endpoints = np.zeros(z.shape, dtype=int)

    z_lo = z < trait_tol
    z_hi = z > 1 - trait_tol
    z_mid = ~(z_lo | z_hi)

    Z_lo = Z < trait_tol
    Z_hi = Z > 1 - trait_tol
    Z_mid = ~(Z_lo | Z_hi)

    endpoints[settled & z_hi  & Z_lo]  = 1
    endpoints[settled & z_hi  & Z_mid] = 2
    endpoints[settled & z_mid & Z_lo]  = 3
    endpoints[settled & z_hi  & Z_hi]  = 4
    endpoints[settled & z_mid & Z_mid] = 5
    endpoints[settled & z_lo  & Z_lo]  = 6
    endpoints[settled & z_mid & Z_hi]  = 7
    endpoints[settled & z_lo  & Z_mid] = 8
    endpoints[settled & z_lo  & Z_hi]  = 9

    endpoint_labels = [
        "unresolved",
        "unspecialised",
        "partial breeder, no helper",
        "no breeder, partial helper",
        "complete breeder, no helper",
        "partial breeder, partial helper",
        "no breeder, complete helper",
        "complete breeder, partial helper",
        "partial breeder, complete helper",
        "complete breeder, complete helper",
    ]

    colors = [
        [0.70, 0.70, 0.70],
        [1.00, 1.00, 1.00],
        [1.00, 0.75, 0.45],
        [0.55, 0.75, 0.85],
        [0.90, 0.45, 0.00],
        [0.85, 0.35, 0.35],
        [0.00, 0.45, 0.70],
        [0.65, 0.30, 0.00],
        [0.00, 0.30, 0.50],
        [0.55, 0.30, 0.60],
    ]

    # Match plotting orientation to parameter order
    varying = [k for k in param_ranges if k in axes]

    if varying[0] == axes[0]:
        endpoints = endpoints.T
        breeder_facilitates = breeder_facilitates.T
        helper_facilitates = helper_facilitates.T

    if ax is None:
        fig, ax = plt.subplots(figsize=(3.5, 3))
        standalone = True
    else:
        standalone = False

    extent = [
        min(param_ranges[axes[0]]), max(param_ranges[axes[0]]),
        min(param_ranges[axes[1]]), max(param_ranges[axes[1]])
    ]

    image = ax.imshow(
        endpoints,
        origin="lower",
        extent=extent,
        aspect="auto",
        interpolation="nearest",
        cmap=ListedColormap(colors),
        norm=BoundaryNorm(np.arange(-0.5, 10.5), 10)
    )

    # Facilitation overlays
    x = np.linspace(extent[0], extent[1], endpoints.shape[1])
    y = np.linspace(extent[2], extent[3], endpoints.shape[0])

    ax.contourf(
        x, y, breeder_facilitates.astype(int),
        levels=[0.5, 1.5],
        colors="none",
        hatches=["///"]
    )

    ax.contourf(
        x, y, helper_facilitates.astype(int),
        levels=[0.5, 1.5],
        colors="none",
        hatches=["\\\\"]
    )

    ax.set_xlabel(labels[axes[0]])
    ax.set_ylabel(labels[axes[1]])

    if standalone:
        from matplotlib.patches import Patch

        cbar = fig.colorbar(image, ax=ax, ticks=range(10))
        cbar.ax.set_yticklabels(endpoint_labels)

        ax.legend(
            handles=[
                Patch(facecolor="white", edgecolor="black",
                      hatch="///", label="Breeder facilitates helper"),
                Patch(facecolor="white", edgecolor="black",
                      hatch="\\\\", label="Helper facilitates breeder")
            ],
            loc="upper left",
            bbox_to_anchor=(1, 1.3)
        )

        plt.show()

    return image

In [95]:
# @title Example endpoint classification

# Example 1

# background = {
#     'r': 0.6,
#     's_season': 0.8,
#     'T': 4,
#     'P0': 10,
#     'Pd': 1.4,
# }

# axes = ('c', 'n')

# z_end, Z_end, settled, breeder_facilitates, helper_facilitates = (
#     calc_numerical_endpoints(axes, background)
# )

# breeder_facilitates=np.zeros_like(z_end, dtype=bool)
# helper_facilitates=np.zeros_like(z_end, dtype=bool)

# plot_numerical_endpoint_classification(
#     z_end, Z_end, settled,
#     breeder_facilitates, helper_facilitates, axes
# )

# # Example 2 - every combination

# background = {
#     'n': 20,
#     'c': 0.5,
#     'r': 0.6,
#     's_season': 0.8,
#     'T': 4,
#     'P0': 10,
#     'Pd': 1.4,
# }

# for axes in it.combinations(('s_season', 'c', 'r', 'n', 'T'), 2):

#     z_end, Z_end, settled, breeder_facilitates, helper_facilitates = (
#         calc_numerical_endpoints(axes, background)
#     )

#     breeder_facilitates=np.zeros_like(z_end, dtype=bool)
#     helper_facilitates=np.zeros_like(z_end, dtype=bool)

#     plot_numerical_endpoint_classification(
#         z_end, Z_end, settled,
#         breeder_facilitates, helper_facilitates, axes
#     )

In [96]:
#@title plot_numerical_endpoint_classification_big

def plot_numerical_endpoint_classification_big(z, Z, settled, axes, trait_tol=1e-5, ax=None):

    endpoints = np.zeros(z.shape, dtype=int)

    z_lo = z < trait_tol
    z_hi = z > 1 - trait_tol
    z_mid = ~(z_lo | z_hi)

    Z_lo = Z < trait_tol
    Z_hi = Z > 1 - trait_tol
    Z_mid = ~(Z_lo | Z_hi)

    endpoints[settled & z_hi  & Z_lo]  = 1
    endpoints[settled & z_hi  & Z_mid] = 2
    endpoints[settled & z_mid & Z_lo]  = 3
    endpoints[settled & z_hi  & Z_hi]  = 4
    endpoints[settled & z_mid & Z_mid] = 5
    endpoints[settled & z_lo  & Z_lo]  = 6
    endpoints[settled & z_mid & Z_hi]  = 7
    endpoints[settled & z_lo  & Z_mid] = 8
    endpoints[settled & z_lo  & Z_hi]  = 9

    colors = [
        [0.70, 0.70, 0.70],
        [1.00, 1.00, 1.00],
        [1.00, 0.75, 0.45],
        [0.55, 0.75, 0.85],
        [0.90, 0.45, 0.00],
        [0.85, 0.35, 0.35],
        [0.00, 0.45, 0.70],
        [0.65, 0.30, 0.00],
        [0.00, 0.30, 0.50],
        [0.55, 0.30, 0.60],
    ]

    varying = [k for k in param_ranges if k in axes]

    if varying[0] == axes[0]:
        endpoints = endpoints.T

    if ax is None:
        fig, ax = plt.subplots(figsize=(3.5, 3))

    image = ax.imshow(
        endpoints,
        origin="lower",
        extent=[
            min(param_ranges[axes[0]]), max(param_ranges[axes[0]]),
            min(param_ranges[axes[1]]), max(param_ranges[axes[1]])
        ],
        aspect="auto",
        interpolation="nearest",
        cmap=ListedColormap(colors),
        norm=BoundaryNorm(np.arange(-0.5, 10.5), 10)
    )

    ax.set_xlabel(labels[axes[0]])
    ax.set_ylabel(labels[axes[1]])

    return image

In [97]:
#@title Panel plot endpoint classification

background = {
    'n': 20,
    'c': 0.5,
    'r': 0.6,
    's_season': 0.8,
    'T': 4,
    'P0': 10,
    'Pd': 1.4,
}

# combinations = list(it.combinations(('s_season', 'c', 'r', 'n'), 2))

# results = [
#     calc_numerical_endpoints(axes, background)
#     for axes in combinations
# ]

# for show_facilitation in [True, False]:

#     fig, axs = plt.subplots(2, 3, figsize=(12, 6), constrained_layout=True)

#     for ax, axes, result in zip(axs.flat, combinations, results):

#         z_end, Z_end, settled, breeder_facilitates, helper_facilitates = result

#         if not show_facilitation:
#             breeder_facilitates = np.zeros_like(z_end, dtype=bool)
#             helper_facilitates = np.zeros_like(z_end, dtype=bool)

#         image = plot_numerical_endpoint_classification(
#             z_end, Z_end, settled,
#             breeder_facilitates, helper_facilitates, axes,
#             ax=ax
#         )

#     cbar = fig.colorbar(image, ax=axs, ticks=range(10), shrink=0.8)
#     cbar.ax.set_yticklabels(endpoint_labels)

#     plt.show()

# Sensitivity

In [98]:
#@title sample_parameters

def sample_parameters(count=10000):
    '''
    Uniform sample across param_ranges using Latin hypercube sampling.
    '''

    sample = qmc.LatinHypercube(d=len(param_ranges), seed=41).random(n=count)
    samples = {}

    for i, (name, values) in enumerate(param_ranges.items()):

        lower, upper = values[0], values[-1]

        if name in ('n', 'T'):
            samples[name] = lower + np.floor((upper - lower + 1) * sample[:, i]).astype(int)
        else:
            samples[name] = lower + (upper - lower) * sample[:, i]

    return samples

In [99]:
#@title analytical_joint_conditions

def analytical_joint_conditions(parameters):
    """Evaluate analytical conditions for complete joint specialisation."""

    # Equation for right breeder nullcline, z+

    a, b, c = sp.Poly(WX, z).all_coeffs()

    zplus = (-b - sp.sqrt(b**2 - 4*a*c)) / (2*a)

    zplus = sp.lambdify(symbols, zplus, 'numpy')

    with np.errstate(invalid='ignore'):
        zplus = zplus(*parameters)

    # Equation for helper nullcline along Z = 0, z0

    z0 = sp.solve(Wx.subs(Z, 0), z)[0]

    z0 = sp.lambdify(symbols, z0, 'numpy')

    z0 = z0(*parameters)

    # Conditions

    helper_not_disfavoured_at_end = Wx_fun(0, 1, *parameters) <= 0

    breeder_not_initially_favoured = WX_fun(1, *parameters) <= 0

    helper_facilitates_breeder = (
        (zplus > 0) &
        (zplus <= 1) &
        (z0 < zplus)
    )

    # A implies B is equivalent to (not A) or B
    return helper_not_disfavoured_at_end & (
         ~breeder_not_initially_favoured | helper_facilitates_breeder
    )

In [100]:
#@title sensitivity

def sensitivity(background):
    """
    Primary effect: count gains and losses of complete joint specialisation
    as one parameter increases, across many parameter backgrounds.

    Pairwise interaction: compare these gains and losses when a second
    parameter has low, middle or high values.
    """

    results = {}

    for parameter, values in param_ranges.items():

        joint = []

        for value in values:

            trial = background.copy()
            trial[parameter] = np.full(background[parameter].shape, value)

            params = prepare_parameters(trial, grid=False)
            joint.append(analytical_joint_conditions(params))

        joint = np.array(joint, dtype=bool)

        # Primary effect
        gain = ~joint[:-1] & joint[1:]
        loss = joint[:-1] & ~joint[1:]

        scale = 100 * ((values.max() - values.min()) / np.diff(values))

        # Secondary effect
        interactions = {}

        for other in param_ranges:

            if parameter == other:
                continue

            boundaries = np.quantile(background[other], [1/3, 2/3])

            terciles = {
                'low': background[other] <= boundaries[0],
                'middle': (background[other] > boundaries[0]) & (background[other] <= boundaries[1]),
                'high': background[other] > boundaries[1]
            }

            interactions[other] = {}

            for level, tercile in terciles.items():

                interactions[other][level] = {'net':
                 (gain[:, tercile].mean(axis=1) - loss[:, tercile].mean(axis=1)) * scale
                }

        results[parameter] = {
            'gain': gain.mean(axis=1) * scale,
            'loss': loss.mean(axis=1) * scale,
            'net': (gain.mean(axis=1) - loss.mean(axis=1)) * scale,
            'interactions': interactions
        }

    return results

In [101]:
#@title plot_sensitivity

def plot_sensitivity(results, net):
    """Plot primary sensitivities on the diagonal and pairwise effects elsewhere."""

    param_ranges = {
        k: v for k, v in globals()['param_ranges'].items()
        if k not in ('T', 'Pd', 'P0')
    }

    param_num = len(param_ranges)

    linestyles = {
        'low': ':',
        'middle': '--',
        'high': '-'
    }

    fig, axes = plt.subplots(
        param_num, param_num,
        figsize=(2.5 * param_num, 2.5 * param_num),
        sharex='col',
        sharey=False
    )

    for col, parameter in enumerate(param_ranges):

        values = param_ranges[parameter]
        midpoints = (values[:-1] + values[1:]) / 2

        for row, other in enumerate(param_ranges):

            ax = axes[row, col]

            if parameter == other:

                # Primary sensitivity
                if net:
                    ax.plot(
                        midpoints,
                        results[parameter]['gain'] - results[parameter]['loss'],
                        color='black',
                        label='net'
                    )

                else:
                    ax.plot(
                        midpoints,
                        results[parameter]['gain'],
                        color='tab:blue',
                        label='gain'
                    )

                    ax.plot(
                        midpoints,
                        results[parameter]['loss'],
                        color='tab:red',
                        label='loss'
                    )

            else:

                # Pairwise effects
                for level, effects in results[parameter]['interactions'][other].items():

                    ax.plot(
                        midpoints,
                        effects['net'],
                        linestyle=linestyles[level],
                        color='black',
                        label=level
                    )

            ax.axhline(0, color='black', linewidth=0.6)
            ax.tick_params(direction='out')

            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)

            if row == 0:
                ax.set_xlabel(labels[parameter])
                ax.xaxis.set_label_position('top')

            if col == 0:
                ax.set_ylabel(labels[other])

    # Primary-effect legend
    handles1, labels1 = axes[0, 0].get_legend_handles_labels()
    handles2, labels2 = axes[1, 0].get_legend_handles_labels()

    fig.legend(
        handles1 + handles2,
        labels1 + labels2,
        loc='lower center',
        bbox_to_anchor=(0.5, 0.05),
        ncol=5,
        frameon=True,
        edgecolor='black'
    )

    # Format-dependent spacing
    if format == "presentation":
        left = 0.18
    else:
        left = 0.16

    fig.subplots_adjust(
        left=left,
        right=0.98,
        bottom=0.10,
        top=0.88,
        wspace=0.35,
        hspace=0.22
    )

    fig.supxlabel(
        "Varied parameter value",
        y=0.93
    )

    fig.supylabel(
        "Range-normalised transitions (percentage points)",
        x=0.035
    )

    fig.savefig(
        "sensitivity_combined.svg",
        bbox_inches='tight'
    )

    plt.show()

In [ ]:
# @title Run Sensitivity

load = False

if load:
    with open("sensitivity.pkl", "rb") as f:
        samples, results = pickle.load(f)

else:

    # run
    samples = sample_parameters(count=int(1e6))
    results = sensitivity(samples)

    # save
    with open("sensitivity.pkl", "wb") as f:
        pickle.dump((samples, results), f)

plot_sensitivity(results, net=False)